In [ ]:
'''
    Code to create illustrations (heatmaps) of OR and AR variations across one-way and two contexts
'''
import numpy as np
import pandas as pd

from options.options import Options
import util.util as util
import util.pre_process as pre

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker

phen_label = 'AFIB' # Choose phenotype to illustrate

opt = Options()
opt.initialize()

phen_title = opt.phen_params[phen_label]['title']
emerge_cutoff = opt.phen_params[phen_label]['emerge_cut']

covars_df = pd.read_csv(opt.env_file, sep='\t', index_col='IID', nrows=5)
bin_defs = opt.bin_defs
bounds, labels, label_map = util.create_bins_and_indices(covars_df, bin_defs)

n = len(opt.labels)

n_bins = len(opt.labels)
ltril_inds = np.tril_indices(n_bins, -1)

# Load files

In [ ]:
all_results = np.load(f'results/{phen_label}/all_results.npy', allow_pickle=True).item()
odds_ratio = np.load(f'results/{phen_label}/top_or.npy')
abs_risk = np.load(f'results/{phen_label}/top_ar.npy')

three_way = np.load(f'results/{phen_label}/results_3way.npy', allow_pickle=True).item()

pdiff = np.load(f'results/{phen_label}/or_pdiff_2way.npy')
pdiff_pval_zero = np.load(f'results/{phen_label}/pval_zero.npy')
pdiff_pval_parent = np.load(f'results/{phen_label}/pval_parent.npy')
bootstrap = np.load(f'results/{phen_label}/bootstrap.npy', allow_pickle=True).item()

# Odds Ratio Heatmap

In [ ]:
odds_all = all_results['top_or']
odds_2way = odds_ratio[:,:]
odds_1way = np.diag(odds_ratio[:,:])

n_bins = odds_2way.shape[0]
heatmap_arr = np.full((n_bins, n_bins+2), np.nan)

heatmap_arr[:, 0] = odds_1way
heatmap_arr[:, 2:] = np.tril(odds_2way, k=-1)

heatmap_arr = np.vstack([heatmap_arr, np.full((2, heatmap_arr.shape[1]), np.nan)])
heatmap_arr[-1, 0] = odds_all

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(heatmap_arr, dtype=bool), k=2)

# Plot heatmap with mask
plt.figure(figsize=(20,20))
ax = sns.heatmap(heatmap_arr[:,:-3], mask=mask[:,:-3], annot=True, fmt=".2f", cmap="coolwarm", center=odds_all,
                 xticklabels=['' for _ in range(len(opt.labels))],
                 yticklabels=opt.labels,
                 square=True, cbar_kws={'shrink': 0.6, 'pad': 0.001}, annot_kws={"size": 20})

# Adjust cbar text properties
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=22, width=2)  # Make text thicker and increase bar thickness
cbar.set_label(f'Odds Ratio', size=28) 

for label in ax.get_xticklabels():
    label.set_fontweight('bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')

ax.hlines(0, 0, 1, color="black", linewidth=9)
ax.hlines(n, 0, 1, color="black", linewidth=5)
ax.vlines(0, 0, n, color="black", linewidth=9)
ax.vlines(1, 0, n, color="black", linewidth=5)

ax.vlines(2, bounds[1], n, color="black", linewidth=5)
for b in bounds[:-1]:
    ax.hlines(b, 0, 1, color="black", linewidth=5)
    ax.hlines(b, 2, b+2, color="black", linewidth=5)
    if b>0:
        ax.vlines(b+2, b, n, color="black", linewidth=5)
ax.hlines(n, 2, b+2, color="black", linewidth=5)

ax.tick_params(axis='y', labelsize=24)
ax.tick_params(axis='x', labelsize=20)
plt.tick_params(left=False, bottom=False)
for idx, label in enumerate(['All','']+opt.labels[:-3]):
    if idx < 1:
        y_offset = n+1
    else:
        y_offset = n-1
    ax.text(idx + 0.5, 1.1 + y_offset, label, 
            ha='right', va='top', fontsize=24, 
            rotation=60, fontweight='bold')
    '''
    ax.text(idx + 0.5, 1.25 + y_offset, label, 
            ha='center', va='top', fontsize=20, 
            rotation=90, fontweight='bold')
    '''

ax.text(
    0.575, .97, 
    f'{phen_title}: Odds Ratios Across Contexts', 
    fontsize=42, 
    fontweight='bold', 
    ha='center', 
    transform=ax.transAxes
)
ax.text(
    0.575, .92, 
    f'(UKB; Eur. Ancestry; Top {int(100*emerge_cutoff)}% of PGS)', 
    fontsize=42, 
    ha='center', 
    transform=ax.transAxes
)

plt.tight_layout()
file_nm = 'or_heatmap.png'
file_nm = f'results/{phen_label}/{file_nm}'
plt.savefig(file_nm, bbox_inches='tight')
plt.show()

# Absolute Risk Heatmap

In [ ]:
ar_all = 100*all_results['top_ar']
ar_2way = 100*abs_risk[:,:]
ar_1way = 100*np.diag(abs_risk[:,:])

n_bins = ar_2way.shape[0]
heatmap_arr = np.full((n_bins, n_bins+2), np.nan)

heatmap_arr[:, 0] = ar_1way
heatmap_arr[:, 2:] = np.tril(ar_2way, k=-1)

heatmap_arr = np.vstack([heatmap_arr, np.full((2, heatmap_arr.shape[1]), np.nan)])
heatmap_arr[-1, 0] = ar_all

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(heatmap_arr, dtype=bool), k=2)

# Plot heatmap with mask
plt.figure(figsize=(20,20))
ax = sns.heatmap(heatmap_arr[:,:-3], mask=mask[:,:-3], annot=True, fmt=".1f", cmap="coolwarm", center=ar_all,
                 xticklabels=['' for _ in range(n)],
                 yticklabels=opt.labels,
                 square=True, cbar_kws={'shrink': 0.6, 'pad': 0.001}, annot_kws={"size": 20})

# Adjust cbar text properties
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=22, width=2)  # Make text thicker and increase bar thickness
cbar.set_label(f'Absolute Risk (%)', size=28) 

ax.tick_params(axis='y', labelsize=24)
ax.tick_params(axis='x', labelsize=20)
for label in ax.get_xticklabels():
    label.set_fontweight('bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')

ax.hlines(0, 0, 1, color="black", linewidth=9)
ax.hlines(n, 0, 1, color="black", linewidth=5)
ax.vlines(0, 0, n, color="black", linewidth=9)
ax.vlines(1, 0, n, color="black", linewidth=5)

ax.vlines(2, bounds[1], n, color="black", linewidth=5)
for b in bounds[:-1]:
    ax.hlines(b, 0, 1, color="black", linewidth=5)
    ax.hlines(b, 2, b+2, color="black", linewidth=5)
    if b>0:
        ax.vlines(b+2, b, n, color="black", linewidth=5)
ax.hlines(n, 2, b+2, color="black", linewidth=5)

plt.tick_params(left=False, bottom=False)
for idx, label in enumerate(['All','']+opt.labels[:-3]):
    if idx < 1:
        y_offset = n+1
    else:
        y_offset = n-1
    
    '''
    ax.text(idx + 0.5, 
            1.25 + y_offset, label, 
            ha='center', va='top', fontsize=20, 
            rotation=90, fontweight='bold')
    '''
    
    ax.text(idx + 0.5, 1.1 + y_offset, label, 
            ha='right', va='top', fontsize=24, 
            rotation=60, fontweight='bold')

ax.text(
    0.575, .97, 
    f'{phen_title}: Absolute Risk Across Contexts', 
    fontsize=42, 
    fontweight='bold', 
    ha='center', 
    transform=ax.transAxes
)
ax.text(
    0.575, .92, 
    f'(UKB; Eur. Ancestry; Top {int(100*emerge_cutoff)}% of PGS)', 
    fontsize=42, 
    ha='center', 
    transform=ax.transAxes
)

plt.tight_layout()
file_nm = 'ar_heatmap.png'
file_nm = f'results/{phen_label}/{file_nm}'
plt.savefig(file_nm, bbox_inches='tight')
plt.show()

# Maximum OR %Δ Heatmap

In [ ]:
def format_for_heatmap(mat):
    heatmap = mat
    diagonal = np.diag(heatmap)

    lower_triangle_mask = np.triu(np.ones_like(mat, dtype=bool), k=0)

    heatmap = np.where(lower_triangle_mask, np.nan, heatmap)
    blank_columns = np.full((heatmap.shape[0], 2), np.nan)
    heatmap = np.hstack((blank_columns, heatmap))

    heatmap[:,0] = diagonal

    return heatmap

xlabels = ['All', '', 'Sex', 'Age', 'Alcohol Intake', 'Smoking History', 'House. Income', 'Towns. Depriv.']
n_vars = pdiff.shape[0]
n_intersects = (n_vars)*(n_vars+1)//2

pdiff_heatmap = format_for_heatmap(pdiff[:,:,0])
pdiff_pval_zero_heatmap = format_for_heatmap(pdiff_pval_zero)
pdiff_pval_parent_heatmap = format_for_heatmap(pdiff_pval_parent)

plt.figure(figsize=(10, 8))
ax = sns.heatmap(
    pdiff_heatmap[:,:-1],
    cmap="Reds",
    annot=True,
    fmt=".2f",
    cbar=True,
    xticklabels=xlabels[:-1],
    yticklabels=xlabels[2:],
    mask=np.isnan(pdiff_heatmap[:, :-1]),
    cbar_kws={'shrink': 0.8, 'label': f'Max. OR %Δ'},
    annot_kws={"size": 18}
)
text_colors = [text.get_color() for text in ax.texts]

pval_zero_thresh = [0.05/n_intersects, 0.01 / n_intersects, 0.001 / n_intersects]
p_div_parent = 1 

ij = 0
for (i, j), val in np.ndenumerate(pdiff_pval_zero_heatmap):
    zero_val_ = pdiff_pval_zero_heatmap[i,j]
    zero_val_parent1_ = pdiff_pval_zero_heatmap[i,0]
    zero_val_parent2_ = pdiff_pval_zero_heatmap[j-2,0]
    
    parent_val_ = pdiff_pval_parent_heatmap[i,j]
    
    if not np.isnan(zero_val_):  # Skip NaN value
        
        pval_annot = ''
        if zero_val_ < pval_zero_thresh[2]:
            pval_annot  += '***'
        elif zero_val_ < pval_zero_thresh[1]:
            pval_annot  += '**'
        elif zero_val_ < pval_zero_thresh[0]:
            pval_annot  += '*'

        add_str = ''
        if parent_val_<.05:
            add_str  = '⨂'
        
        ax.text(j + 0.5, i + 0.25, pval_annot, ha='center', va='center', color=text_colors[ij], fontsize=18) # have this white if the text annotated in square is white as well
        ax.text(j + 0.5, i + 0.75, add_str, ha='center', va='center', color=text_colors[ij], fontsize=14) # have this white if the text annotated in square is white as well

        ij += 1

#title = f'{opt.phen_params[phen_label]["title"]}: UKB (Eur. Ancestry)'
ax.text(
    0.5, .965+.075, 
    f'{phen_title}: Maximum OR %Δ Across Contexts', 
    fontsize=24, 
    fontweight='bold', 
    ha='center', 
    transform=ax.transAxes
)
ax.text(
    0.6, .9+.075, 
    f'(UKB; Eur. Ancestry; Top {int(100*emerge_cutoff)}% of PGS)', 
    fontsize=20, 
    ha='center', 
    transform=ax.transAxes
)

#plt.title(title, fontsize=18)  # Increase title font size
plt.xticks(fontsize=18)  # Increase x-axis labels font size
plt.yticks(fontsize=18)  # Increase y-axis labels font size

cbar = ax.collections[0].colorbar
cbar.ax.yaxis.label.set_size(18)

cbar.ax.tick_params(labelsize=18)
cbar.ax.yaxis.label.set_size(18)

plt.tight_layout()
file_nm = 'or_pdiff_heatmap.png'
file_nm = f'results/{phen_label}/{file_nm}'
plt.savefig(file_nm, bbox_inches='tight')
plt.show()

In [ ]:
def find_bootstrap(bootstrap, idx):
    for key in bootstrap.keys():
        for extreme in ['min', 'max']:
            if bootstrap[key][extreme]['index'] == idx:
                return bootstrap[key][extreme]['bootstrap']['top_or']

max_1way_idx = np.argmax(np.diagonal(odds_ratio))
min_1way_idx = np.argmin(np.diagonal(odds_ratio))
max_1way_coords, min_1way_coords = (max_1way_idx, max_1way_idx), (min_1way_idx, min_1way_idx)
max_1way_value = odds_ratio[max_1way_coords]
min_1way_value = odds_ratio[min_1way_coords]
max_1way = find_bootstrap(bootstrap, max_1way_coords)
min_1way= find_bootstrap(bootstrap, min_1way_coords)
label_max_1way = opt.labels_shorthand[max_1way_idx]
label_min_1way = opt.labels_shorthand[min_1way_idx]

# 2-way analysis
max_2way_idx = np.nanargmax(odds_ratio[ltril_inds])
min_2way_idx = np.nanargmin(odds_ratio[ltril_inds])
max_2way_coords = (ltril_inds[0][max_2way_idx], ltril_inds[1][max_2way_idx])
min_2way_coords = (ltril_inds[0][min_2way_idx], ltril_inds[1][min_2way_idx])
max_2way_value = odds_ratio[max_2way_coords]
min_2way_value = odds_ratio[min_2way_coords]
max_2way = find_bootstrap(bootstrap, max_2way_coords)
min_2way = find_bootstrap(bootstrap, min_2way_coords)
label_max_2way = f'{opt.labels_shorthand[max_2way_coords[1]]} &\n{opt.labels_shorthand[max_2way_coords[0]]}'
label_min_2way = f'{opt.labels_shorthand[min_2way_coords[1]]} &\n{opt.labels_shorthand[min_2way_coords[0]]}'

# 3-way analysis
odds_3way = np.load(f'results/{phen_label}/minmax_3way.npy', allow_pickle=True).item()
max_3way_idx = odds_3way['max']['index']
min_3way_idx = odds_3way['min']['index']
max_3way = odds_3way['max']['top_or']
min_3way = odds_3way['min']['top_or']

label_max_3way = f'{opt.labels_shorthand[max_3way_idx[2]]} &\n{opt.labels_shorthand[max_3way_idx[1]]} &\n{opt.labels_shorthand[max_3way_idx[0]]}'
label_min_3way = f'{opt.labels_shorthand[min_3way_idx[2]]} &\n{opt.labels_shorthand[min_3way_idx[1]]} &\n{opt.labels_shorthand[min_3way_idx[0]]}'

_, p_one = util.calc_empirical_pval(max_1way-min_1way, two_tail=False)
_, p_two = util.calc_empirical_pval(max_2way-min_2way, two_tail=False)
_, p_three = util.calc_empirical_pval(max_3way-min_3way, two_tail=False)

In [ ]:
x_range = (0.95, 8)
n_bins = 50

# Define the figure and axes
fig, axes = plt.subplots(3, 1, figsize=(5, 14))

# Distribution histogram for one_low and one_high
axes[0].hist(min_1way[1:], bins=n_bins, alpha=0.6, color='skyblue', label=label_min_1way, edgecolor='black', range=x_range)
axes[0].hist(max_1way[1:], bins=n_bins, alpha=0.6, color='orange', label=label_max_1way, edgecolor='black', range=x_range)
axes[0].set_title('One-Way Intersection', fontsize=20)
axes[0].legend(fontsize=16, loc='upper right', title=f'p ≈ {p_one}', title_fontsize=16)
axes[0].tick_params(axis='both', which='major', labelsize=16)
axes[0].grid(axis='x', linestyle='--', alpha=1)
axes[0].set_ylabel('Frequency', fontsize=22)


# Distribution histogram for two_low and two_high
axes[1].hist(min_2way[1:], bins=n_bins, alpha=0.6, color='skyblue', label=label_min_2way, edgecolor='black', range=x_range)
axes[1].hist(max_2way[1:], bins=n_bins, alpha=0.6, color='orange', label=label_max_2way, edgecolor='black', range=x_range)
axes[1].set_title('Two-Way Intersection', fontsize=20)
axes[1].legend(fontsize=16, loc='center right', title=f'p ≈ {p_two}', title_fontsize=16)
axes[1].tick_params(axis='both', which='major', labelsize=16)
axes[1].grid(axis='x', linestyle='--', alpha=1)
axes[1].set_ylabel('Frequency', fontsize=22)


# Distribution histogram for three_low and three_high
axes[2].hist(min_3way[1:], bins=n_bins, alpha=0.6, color='skyblue', label=label_min_3way, edgecolor='black', range=x_range)
axes[2].hist(max_3way[1:], bins=n_bins, alpha=0.6, color='orange', label=label_max_3way, edgecolor='black', range=x_range)
axes[2].set_title('Three-Way Intersection', fontsize=20)
axes[2].legend(fontsize=16, loc='upper right', title=f'p ≈ {p_three}', title_fontsize=16)
axes[2].tick_params(axis='both', which='major', labelsize=16)
axes[2].grid(axis='x', linestyle='--', alpha=1)
axes[2].set_ylabel('Frequency', fontsize=22)

axes[2].set_xlabel(f"Adj. Odds Ratio\n(UKB; Eur.; top {int(100*opt.phen_params[phen_label]['emerge_cut'])}%)", fontsize=20)
title_str = f"{phen_label}:\n"
fig.suptitle(title_str+'Low & High OR Contexts\nBootstrap Distributions', fontsize=24, y=.99, fontweight='bold',)

fig.tight_layout()
fig.savefig(f'results/{phen_label}/bootstrap_extremes.png', dpi=300, bbox_inches='tight')
plt.show
